# 08 - Fabric capacity benchmark and 30-day feasibility gate

Run this notebook through a benchmark pipeline over representative videos and increasing `CONCURRENT_WORKERS` levels. One activity measures one worker; the pipeline supplies concurrency. Results calculate the minimum workers required to process 200,000 video-hours in 30 days. Do not approve a capacity from a single short video or an interactive notebook run.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

## Before you run

This notebook has two modes: many **worker activities** write benchmark rows, then one **gate activity** evaluates exactly that batch. Do not use an interactive single-video run to approve capacity.

### 1. Prepare Fabric and representative videos

1. Run `00_bootstrap_lakehouse.ipynb` so the `people_counter_processing_benchmarks` table exists.
2. Attach and pin the same default Lakehouse, published Environment, and Fabric runtime that production workers will use. The notebook supports Fabric-native CPU execution only.
3. Select at least three combined duration/resolution/motion buckets: an easy case, the common case, and a long or high-motion case. Include materially expensive codecs or camera families. The sustained batch should resemble the approved source inventory rather than contain only easy clips.
4. Record each video's exact source duration in seconds. Use media metadata or a trusted probe; do not use processing time or sampled duration.

### 2. Run a sustained benchmark batch

Create the `pc-capacity-benchmark` Data pipeline described in section 7.4 of `notebooks/fabric/README.md`. Its `BENCHMARK_ITEMS` Array parameter contains objects with `video_uri`, `sample_name`, and `duration_seconds`. Configure a parallel `ForEachBenchmarkItems` with Items `@pipeline().parameters.BENCHMARK_ITEMS`, a literal Batch count equal to `CONCURRENT_WORKERS`, and one child Notebook activity named `RunBenchmarkWorker`.

Map the child video's parameters from `@item().video_uri`, `@item().sample_name`, and `@item().duration_seconds`. Place a separate `EvaluateCapacityGate` Notebook activity outside the ForEach, connect it with **On completion**, and set its `EXPECTED_BATCH_MEMBERS` to `@length(pipeline().parameters.BENCHMARK_ITEMS)`. Keep the selected concurrency busy for at least six hours. One worker activity invocation is one recorded batch member. Use one unique `BENCHMARK_BATCH_ID` for the entire run and a new ID for each concurrency, configuration, or rerun.

For every worker activity set:

- `RUN_INFERENCE=true`
- `ENFORCE_CAPACITY_GATE=false`
- `BENCHMARK_BATCH_ID` to the shared batch ID, for example `prod-f64-c08-20260924-r01`
- `CONCURRENT_WORKERS` to the pipeline concurrency being tested, for example `8`; this is **not** the total number of activity instances
- `EXPECTED_BATCH_MEMBERS=0`
- `VIDEO_URI`, `SAMPLE_NAME`, and `EXPECTED_VIDEO_DURATION_SECONDS` for that activity's video
- all capacity, runtime, model, and target parameters to the same values across the batch

Record the total number of activity instances launched, including failures. A retry creates another row. If a member is retried or accidentally duplicated, discard that batch ID and rerun rather than changing the expected count to make the gate pass.

### 3. Run the gate activity

After every worker activity has finished, run this notebook once with:

- `RUN_INFERENCE=false`
- `ENFORCE_CAPACITY_GATE=true`
- the same `BENCHMARK_BATCH_ID`, `CAPACITY_SKU`, `RUNTIME_VERSION`, `CONCURRENT_WORKERS`, and model configuration
- `EXPECTED_BATCH_MEMBERS` equal to the total activity instances launched
- `VIDEO_URI=""`, `SAMPLE_NAME=""`, and `EXPECTED_VIDEO_DURATION_SECONDS=0.0`; these are unused in gate mode

The gate fails unless the filtered batch has exactly the expected rows, no failed rows, a wall-clock span of at least six hours, and sufficient observed aggregate throughput. Make pipeline promotion depend on this gate activity succeeding.

## Parameter worksheet

| Parameter | Worker activity value | Gate activity value / meaning |
|---|---|---|
| `RUN_INFERENCE` | `true` | `false`; reads existing rows only |
| `ENFORCE_CAPACITY_GATE` | `false` | `true`; raises an error if the gate fails |
| `BENCHMARK_BATCH_ID` | One non-empty ID shared by the sustained run | The same ID; change it for every concurrency/configuration/rerun |
| `VIDEO_URI` | One readable local, Lakehouse, or ABFS video URI | Empty; unused |
| `SAMPLE_NAME` | Stable label such as `common-1080p-medium-motion-01` | Empty; unused |
| `EXPECTED_VIDEO_DURATION_SECONDS` | Exact positive source duration in seconds | `0.0`; unused |
| `CAPACITY_SKU` | Actual assigned SKU label, for example `F64` | Exactly the same label; this grouping key is not auto-detected |
| `RUNTIME_VERSION` | Exact pinned Fabric runtime label shown by the Environment/notebook | Exactly the same label; it is not auto-detected |
| `CONCURRENT_WORKERS` | Actual simultaneous activity limit under test | The same integer; not total batch members |
| `EXPECTED_BATCH_MEMBERS` | `0` | Exact total worker activity instances launched, including failed instances |
| `DATABASE` | Empty for the attached default Lakehouse, otherwise its valid SQL identifier | Same value |
| `TABLE_PREFIX` | `people_counter` unless bootstrap used another prefix | Same value |
| `PIPELINE` | Production choice: `rtdetr-osnet` or `rfdetr-botsort` | Same value |
| `DEVICE_VARIANT` | `cpu` | `cpu`; other values are rejected |
| `DEVICE` | `cpu` | `cpu`; other values are rejected |
| `BATCH_SIZE` | Intended production batch size, at least `1` | Same value |
| `SAMPLE_FPS` | Intended production rate, normally `3.0`; use `None` only if production processes every frame | Same value |
| `DETECTION_THRESHOLD` | Intended production value from `0.1` through `1.0`, normally `0.6` | Same value |
| `USE_FP16` | `false` | `false`; required for the Fabric-native CPU benchmark |
| `LINE` | `[]` without line counting, otherwise `[x1, y1, x2, y2]` in source-video pixels | Same value because it is part of the configuration hash |
| `DETECTOR_MODEL` | `r18` or `r50`; use the production RT-DETR choice and a stable value for RF-DETR | Same value |
| `CAMERA_MOTION_COMPENSATION` | `None` for RT-DETR; for RF-DETR use the intended `true`, `false`, or `None` | Same value |
| `TARGET_VIDEO_HOURS` | Backfill scope, normally `200000.0` | Same value |
| `DEADLINE_DAYS` | Allowed elapsed days, normally `30.0` | Same value |
| `UTILIZATION` | Productive-time fraction in `(0, 1]`, normally `0.80` | Same value; accounts for admission, maintenance, and idle time |
| `HEADROOM_FACTOR` | Retry/data-variance multiplier at least `1.0`, normally `1.20` | Same value; already supplies 20% headroom |

Boolean values may arrive from Fabric as booleans or the strings `true`/`false`. `DATABASE` and `TABLE_PREFIX` must be SQL identifiers. `LINE` may be empty or contain exactly four integer coordinates inside each source frame. Reverse its endpoints if the intended in/out direction is reversed.

## Reading the gate output

- `required_aggregate_speed_x = TARGET_VIDEO_HOURS * HEADROOM_FACTOR / (DEADLINE_DAYS * 24 * UTILIZATION)`. With the defaults this is `416.67x`, meaning 416.67 source-video seconds per wall-clock second.
- `required_workers_with_headroom` divides that required speed by the batch's conservative p10 single-activity speed and rounds up. Treat it as a planning signal; approve only a concurrency that was actually sustained and passed.
- `best_six_hour_aggregate_speed_x` uses total video seconds divided by elapsed time from the earliest member start to the latest completion. It includes staging, inference, orchestration, admission delays, and gaps.
- `observed_batch_members` must equal `expected_batch_members`, and `failed_benchmarks` must be zero.
- `capacity_gate_passed=true` proves throughput only for the exact tested SDK, runtime, SKU label, configuration, and concurrency.

The notebook does not infer Spark cores, memory, or Capacity Unit consumption from `CAPACITY_SKU`. `peak_memory_mb` is not currently populated. For the same six-hour interval, record cores and peak driver/executor memory from Spark/Fabric monitoring and CU seconds, peaks, and throttling from the Fabric Capacity Metrics app. Multiply per-activity cores and memory by the tested concurrency, verify they fit with operating margin, and reject the plan if throughput passes but resource limits, protected workloads, or cost do not.

In [ ]:
RUN_INFERENCE = False
ENFORCE_CAPACITY_GATE = False
BENCHMARK_BATCH_ID = ""
VIDEO_URI = ""
SAMPLE_NAME = ""
EXPECTED_VIDEO_DURATION_SECONDS = 0.0
CAPACITY_SKU = "UNSET"
RUNTIME_VERSION = "UNSET"
CONCURRENT_WORKERS = 1
EXPECTED_BATCH_MEMBERS = 0
DATABASE = ""
TABLE_PREFIX = "people_counter"
PIPELINE = "rtdetr-osnet"
DEVICE_VARIANT = "cpu"
DEVICE = "cpu"
BATCH_SIZE = 1
SAMPLE_FPS = 3.0
DETECTION_THRESHOLD = 0.6
USE_FP16 = False
LINE = []
DETECTOR_MODEL = "r18"
CAMERA_MOTION_COMPENSATION = None
TARGET_VIDEO_HOURS = 200000.0
DEADLINE_DAYS = 30.0
UTILIZATION = 0.80
HEADROOM_FACTOR = 1.20

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from urllib.parse import urlsplit
import hashlib
import json
import math
import os
import re
import time
import uuid

# Fabric can fail inside hf-xet's Reqwest client; use Hugging Face's HTTPS downloader instead.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

import notebookutils
from pyspark.sql import SparkSession, functions as F

from people_counter import RFDetrBotsortConfig, RTDetrOsnetConfig, run


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def parameter_bool(value: object, name: str) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


def optional_parameter_bool(value: object, name: str) -> bool | None:
    if value is None or (isinstance(value, str) and value.strip().lower() in {"", "null", "none"}):
        return None
    return parameter_bool(value, name)


run_inference = parameter_bool(RUN_INFERENCE, "RUN_INFERENCE")
enforce_capacity_gate = parameter_bool(ENFORCE_CAPACITY_GATE, "ENFORCE_CAPACITY_GATE")
benchmark_batch_id = BENCHMARK_BATCH_ID.strip()
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
if int(CONCURRENT_WORKERS) < 1:
    raise ValueError("CONCURRENT_WORKERS must be at least 1")
expected_batch_members = int(EXPECTED_BATCH_MEMBERS)
if expected_batch_members < 0:
    raise ValueError("EXPECTED_BATCH_MEMBERS cannot be negative")
if DEVICE_VARIANT != "cpu" or DEVICE != "cpu":
    raise ValueError("Fabric-native Spark benchmark requires CPU device settings")
if USE_FP16 not in {False, "false", "False"}:
    raise ValueError("Fabric-native CPU benchmark requires USE_FP16=false")
validated_use_fp16 = False
if PIPELINE not in {"rtdetr-osnet", "rfdetr-botsort"} or DETECTOR_MODEL not in {"r18", "r50"}:
    raise ValueError("Benchmark pipeline/model selection is invalid")
validated_batch_size = int(BATCH_SIZE)
validated_sample_fps = None if SAMPLE_FPS is None else float(SAMPLE_FPS)
validated_threshold = float(DETECTION_THRESHOLD)
validated_camera_motion = optional_parameter_bool(
    CAMERA_MOTION_COMPENSATION,
    "CAMERA_MOTION_COMPENSATION",
)
if validated_batch_size < 1:
    raise ValueError("BATCH_SIZE must be at least 1")
if validated_sample_fps is not None and validated_sample_fps <= 0:
    raise ValueError("SAMPLE_FPS must be positive or null")
if not 0.1 <= validated_threshold <= 1:
    raise ValueError("DETECTION_THRESHOLD must be between 0.1 and 1")
if not 0 < float(UTILIZATION) <= 1 or float(HEADROOM_FACTOR) < 1:
    raise ValueError("UTILIZATION and HEADROOM_FACTOR are invalid")
if enforce_capacity_gate and not benchmark_batch_id:
    raise ValueError("BENCHMARK_BATCH_ID is required when ENFORCE_CAPACITY_GATE=true")
if enforce_capacity_gate and expected_batch_members < 1:
    raise ValueError("EXPECTED_BATCH_MEMBERS must be positive when ENFORCE_CAPACITY_GATE=true")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
config_value = {
    "pipeline": PIPELINE,
    "device_variant": DEVICE_VARIANT,
    "device": DEVICE,
    "batch_size": validated_batch_size,
    "sample_fps": validated_sample_fps,
    "detection_threshold": validated_threshold,
    "use_fp16": validated_use_fp16,
    "line": LINE,
    "detector_model": DETECTOR_MODEL,
    "camera_motion_compensation": validated_camera_motion,
}
config_json = json.dumps(config_value, sort_keys=True, separators=(",", ":"))
config_sha256 = hashlib.sha256(config_json.encode("utf-8")).hexdigest()
sdk_version = version("people-counter")

if run_inference:
    if not VIDEO_URI or not SAMPLE_NAME or float(EXPECTED_VIDEO_DURATION_SECONDS) <= 0:
        raise ValueError("VIDEO_URI, SAMPLE_NAME, and EXPECTED_VIDEO_DURATION_SECONDS are required")
    if not benchmark_batch_id:
        raise ValueError("BENCHMARK_BATCH_ID must identify one sustained concurrent test run")
    if not isinstance(LINE, list) or (
        LINE
        and (
            len(LINE) != 4
            or any(isinstance(value, bool) or not isinstance(value, int) for value in LINE)
        )
    ):
        raise ValueError("LINE must be empty or contain four source-video pixel coordinates")
    benchmark_id = uuid.uuid4().hex
    temporary = Path("/tmp") / f"people-counter-benchmark-{benchmark_id}"
    temporary.mkdir(mode=0o700, parents=False, exist_ok=False)
    staged = temporary / (Path(urlsplit(VIDEO_URI).path).name or "sample.video")
    benchmark_started_at = datetime.now(timezone.utc)
    started = time.perf_counter()
    succeeded = False
    error_message = None
    processing_seconds = 0.0
    end_to_end_seconds = 0.0
    speed = 0.0
    try:
        if Path(VIDEO_URI).is_file():
            staged = Path(VIDEO_URI)
        else:
            copied = notebookutils.fs.cp(VIDEO_URI, staged.as_uri())
            if copied is False or not staged.is_file():
                raise RuntimeError(f"Could not stage benchmark video: {VIDEO_URI}")
        common = {
            "video": staged,
            "device_variant": DEVICE_VARIANT,
            "device": DEVICE,
            "batch_size": validated_batch_size,
            "sample_fps": validated_sample_fps,
            "detection_threshold": validated_threshold,
            "use_fp16": validated_use_fp16,
            "line": tuple(LINE) if LINE else None,
        }
        if PIPELINE == "rtdetr-osnet":
            config = RTDetrOsnetConfig(**common, detector_model=DETECTOR_MODEL)
        elif PIPELINE == "rfdetr-botsort":
            config = RFDetrBotsortConfig(
                **common,
                camera_motion_compensation=validated_camera_motion,
            )
        else:
            raise ValueError(f"Unsupported PIPELINE: {PIPELINE}")
        result = run(config)
        processing_seconds = result.processing_seconds
        end_to_end_seconds = time.perf_counter() - started
        speed = float(EXPECTED_VIDEO_DURATION_SECONDS) / end_to_end_seconds
        succeeded = True
    except Exception as error:
        end_to_end_seconds = time.perf_counter() - started
        error_message = f"{type(error).__name__}: {error}"[:4000]
        raise
    finally:
        row = {
            "benchmark_id": benchmark_id,
            "benchmark_batch_id": benchmark_batch_id,
            "benchmark_started_at": benchmark_started_at,
            "completed_at": datetime.now(timezone.utc),
            "capacity_sku": CAPACITY_SKU,
            "runtime_version": RUNTIME_VERSION,
            "sdk_version": sdk_version,
            "config_sha256": config_sha256,
            "sample_name": SAMPLE_NAME,
            "video_duration_seconds": float(EXPECTED_VIDEO_DURATION_SECONDS),
            "end_to_end_seconds": end_to_end_seconds,
            "overhead_seconds": max(0.0, end_to_end_seconds - processing_seconds),
            "processing_seconds": processing_seconds,
            "speed_x_realtime": speed,
            "peak_memory_mb": None,
            "concurrent_workers": int(CONCURRENT_WORKERS),
            "succeeded": succeeded,
            "error_message": error_message,
        }
        spark_session.createDataFrame(
            [row],
            spark_session.table(table("processing_benchmarks")).schema,
        ).write.format("delta").mode("append").saveAsTable(table("processing_benchmarks"))
        if staged.parent == temporary:
            staged.unlink(missing_ok=True)
        temporary.rmdir()
else:
    print("Inference disabled. Set RUN_INFERENCE=true only in the Fabric benchmark pipeline.")

In [ ]:
matching_benchmarks = spark_session.table(table("processing_benchmarks")).where(
    (F.col("config_sha256") == config_sha256)
    & (F.col("capacity_sku") == CAPACITY_SKU)
    & (F.col("runtime_version") == RUNTIME_VERSION)
    & (F.col("sdk_version") == sdk_version)
    & (F.col("concurrent_workers") == int(CONCURRENT_WORKERS))
)
if benchmark_batch_id:
    matching_benchmarks = matching_benchmarks.where(
        F.col("benchmark_batch_id") == benchmark_batch_id
    )
observed_batch_members = matching_benchmarks.count()
failed_benchmarks = matching_benchmarks.where(F.col("succeeded") == False).count()
benchmarks = matching_benchmarks.where(F.col("succeeded") == True)
summary = (
    benchmarks.groupBy("benchmark_batch_id", "concurrent_workers")
    .agg(
        F.count("benchmark_id").alias("samples"),
        F.avg("speed_x_realtime").alias("average_speed_x"),
        F.expr("percentile_approx(speed_x_realtime, 0.10)").alias("p10_speed_x"),
        F.expr("percentile_approx(end_to_end_seconds, 0.95)").alias("p95_end_to_end_seconds"),
        F.sum("video_duration_seconds").alias("total_video_seconds"),
        F.min("benchmark_started_at").alias("batch_started_at"),
        F.max("completed_at").alias("batch_completed_at"),
    )
    .withColumn(
        "sustained_wall_seconds",
        F.unix_timestamp("batch_completed_at") - F.unix_timestamp("batch_started_at"),
    )
    .withColumn(
        "observed_aggregate_speed_x",
        F.col("total_video_seconds") / F.col("sustained_wall_seconds"),
    )
    .orderBy("benchmark_batch_id", "concurrent_workers")
)
display(summary)
rows = summary.collect()
if rows and benchmark_batch_id:
    conservative_speed = min(row.p10_speed_x for row in rows if row.p10_speed_x is not None)
    required_workers = math.ceil(
        float(TARGET_VIDEO_HOURS)
        * float(HEADROOM_FACTOR)
        / (float(DEADLINE_DAYS) * 24.0 * conservative_speed * float(UTILIZATION))
    )
    required_aggregate_speed = (
        float(TARGET_VIDEO_HOURS) * float(HEADROOM_FACTOR)
        / (float(DEADLINE_DAYS) * 24.0 * float(UTILIZATION))
    )
    sustained_rows = [
        row
        for row in rows
        if row.sustained_wall_seconds is not None and row.sustained_wall_seconds >= 6 * 3600
    ]
    best_aggregate_speed = max(
        (row.observed_aggregate_speed_x for row in sustained_rows),
        default=0.0,
    )
    outcome = {
        "target_video_hours": float(TARGET_VIDEO_HOURS),
        "deadline_days": float(DEADLINE_DAYS),
        "benchmark_batch_id": benchmark_batch_id,
        "sdk_version": sdk_version,
        "conservative_speed_x": conservative_speed,
        "required_workers_with_headroom": required_workers,
        "required_aggregate_speed_x": required_aggregate_speed,
        "best_six_hour_aggregate_speed_x": best_aggregate_speed,
        "failed_benchmarks": failed_benchmarks,
        "expected_batch_members": expected_batch_members,
        "observed_batch_members": observed_batch_members,
        "capacity_gate_passed": (
            observed_batch_members == expected_batch_members
            and failed_benchmarks == 0
            and best_aggregate_speed >= required_aggregate_speed
        ),
    }
elif rows:
    outcome = {
        "capacity_gate_passed": False,
        "reason": "BENCHMARK_BATCH_ID is required to evaluate one representative batch",
        "failed_benchmarks": failed_benchmarks,
        "expected_batch_members": expected_batch_members,
        "observed_batch_members": observed_batch_members,
        "sdk_version": sdk_version,
    }
else:
    outcome = {
        "capacity_gate_passed": False,
        "reason": "No matching successful benchmarks for the selected batch and SDK version",
        "benchmark_batch_id": benchmark_batch_id or None,
        "failed_benchmarks": failed_benchmarks,
        "expected_batch_members": expected_batch_members,
        "observed_batch_members": observed_batch_members,
        "sdk_version": sdk_version,
    }
print(json.dumps(outcome, sort_keys=True))
if enforce_capacity_gate and not outcome["capacity_gate_passed"]:
    raise RuntimeError(f"Fabric capacity gate failed: {json.dumps(outcome, sort_keys=True)}")